In [1]:
import numpy as np
import matplotlib.pyplot as plt
import datasets

c:\Users\patog\Documents\Code\ml\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
mnist_ds = datasets.load_from_disk("ylecun/mnist").with_format("numpy")

# 2. Extract contiguous blocks directly (instant PyArrow extraction)
X_train = (mnist_ds["train"]["image"][:] / 255.0).astype(np.float32)
Y_train = mnist_ds["train"]["label"][:].astype(np.int64)

X_test = (mnist_ds["test"]["image"][:] / 255.0).astype(np.float32)
Y_test = mnist_ds["test"]["label"][:].astype(np.int64)

# 3. Fast in-memory shuffle using NumPy
rng = np.random.default_rng(42)

train_idx = rng.permutation(len(X_train))
X_train, Y_train = X_train[train_idx], Y_train[train_idx]

test_idx = rng.permutation(len(X_test))
X_test, Y_test = X_test[test_idx], Y_test[test_idx]

# one hotting
Y_train = np.eye(10, dtype=np.float32)[Y_train]   
Y_test = np.eye(10, dtype=np.float32)[Y_test]     

# flatening and changing resolution
X_train = X_train.reshape(-1, 14, 2, 14, 2).mean(axis=(2, 4)).reshape(len(X_train), -1)  
X_test = X_test.reshape(-1, 14, 2, 14, 2).mean(axis=(2, 4)).reshape(len(X_test), -1)     

In [4]:
from abc import ABC, abstractmethod
from typing import Iterator
from numpy.typing import NDArray

class Trainer():
    def __init__(self, optimizer: Optimizer, x_train_set=NDArray, y_train_set=NDArray , batching=False, chunk_size=-1, batch_size = -1):
        self.optimizer = optimizer
        self.batching = batching
        self.chunk_size = chunk_size
        self.batch_size = batch_size
        self.x_train_set = x_train_set
        self.y_train_set = y_train_set

    def train(self, n_steps=-1, report_freq=1): # each step is composed of n chunks or batch elements
        for i in range(n_steps):
            loss = 0
            for j in range(self.chunk_size):
                loss += self.optimizer.step_accumulate_gradients(self.x_train_set[i * self.chunk_size + j], self.y_train_set[i * self.chunk_size + j])
            self.optimizer.step_learn()

            loss /= self.chunk_size
            if i % report_freq == 0: print(f"step: {i} loss = {loss}")



class Model(ABC):

    layers: list

    @abstractmethod
    def accumulate_gradients(self, gradients_forward): pass
    
    def zero_grad(self):
        for layer in self.layers: layer.zero_grad()

    @abstractmethod
    def __call__(self, x): pass

    @abstractmethod
    def parameters(self) -> Iterator[tuple[NDArray[np.float32], NDArray[np.float32]]]: pass

    
class Optimizer(ABC):
    
    @abstractmethod
    def step_accumulate_gradients(self, x, y) -> float: pass

    @abstractmethod
    def step_learn(self): pass

   


class Layer():
    def __init__(self, nin, nout, nonlin, seed=42): # eah nout is a neuron
        self._tmp = np.zeros((nout, nin), dtype=np.float32) # used as a buffer to avoid reallocating constantly
        self.nout = nout
        self.nin = nin
        rng = np.random.default_rng(seed)

        self.weights = rng.standard_normal((nout, nin), dtype=np.float32) * np.float32(np.sqrt(2.0 / nin)) 
        self.gradients = np.zeros((nout, nin), dtype=np.float32)

        self.bias_weights = rng.standard_normal((nout,), dtype=np.float32) * np.float32(np.sqrt(2.0 / nin)) 
        self.bias_gradients = np.zeros((nout,), dtype=np.float32)

        self.mask = np.ones((nout,), dtype=np.float32)
        self.nonlin = nonlin
        self.activationsin = np.zeros((nin,), dtype=np.float32)
        self.activationsout = np.zeros((nout,), dtype=np.float32)
        

    def __call__(self, activation):
        self.activationsin = activation
        neurons_out = (self.weights @ activation) + self.bias_weights
        if self.nonlin:
            self.mask = neurons_out > 0 # ReLu activation function
        self.activationsout = neurons_out * self.mask
        return self.activationsout
    
    def accumulate_gradients(self, gradients_forward):
        # gradients_forward is of shape [nout]
        # self.gradients is of shape [nout, nin]
        # self.activationsin of shape [nin] -> [nout, nin]

        gradients_forward = gradients_forward * self.mask # kills gradients of dead neurons

        weight_activationsin = np.broadcast_to(self.activationsin, (self.nout, self.nin)) # each weight contains a matching activation
        weight_forward_gradients = np.broadcast_to(gradients_forward[:, np.newaxis], (self.nout, self.nin))

        # a weights local derivative is its corresponding input then we multiply to chain it to/ by the upstream derivative
        np.multiply(weight_activationsin, weight_forward_gradients, out=self._tmp)
        self.gradients += self._tmp

        # bias gradient have a local derivative thats always 1 so we only chain to upstream derivative
        self.bias_gradients += gradients_forward

        # gradient at each input, equals the sum of (each weight connected to it) multiplied by its upstream gradient (forward gradient)
        # self.weights [nout, nin + 1]   gradients_forward [nout]
        gradients_backward = gradients_forward @ self.weights
        return gradients_backward
        

    def zero_grad(self):
        self.gradients.fill(0)
        self.bias_gradients.fill(0)

    # def update_weights(self, learning_rate):
    #     np.multiply(learning_rate, self.gradients, out=self._tmp) # weights
        
    #     self.weights -= self._tmp
    #     self.bias_weights -= self.bias_gradients * learning_rate

    def parameters(self):
        yield self.weights, self.gradients
        yield self.bias_weights, self.bias_gradients

    def __repr__(self):
        nin = self.weights.shape[1]
        nout = self.weights.shape[0]
        
        lines = [f"Layer(nin={nin}, nout={nout}, nonlin={self.nonlin}):"]
        
        # Format weights matrix and bias column
        for i in range(nout):
            # Format each weight with fixed total width (e.g., 7 chars, 3 decimals, space for sign)
            w_row = " ".join(f"{w: 7.3f}" for w in self.weights[i])
            bias = f"{self.bias_weights[i]: 7.3f}"
            mask = f"{int(self.mask[i])}"
            
            lines.append(f"  Neuron {i}: W = [{w_row} ]  b = {bias}  mask = {mask}")
            
        return "\n".join(lines)

    
class MLP(Model):
    def __init__(self, nin, nouts): # int, int[]
        self.layers = []

        # making all layers nonlin (relu) except the last

        if len(nouts) > 1: self.layers.append(Layer(nin=nin, nout=nouts[0], nonlin=True))
        else: self.layers.append(Layer(nin=nin, nout=nouts[0], nonlin=False))

        for i, width in enumerate(nouts[1:-1], start=1):
            self.layers.append(Layer(nin=nouts[i-1], nout=width, nonlin=True, seed=42 + i))

        if len(nouts) > 1: self.layers.append(Layer(nin=nouts[-2], nout=nouts[-1], nonlin=False))

    def __call__(self, activations):
        for layer in self.layers:
            activations = layer(activations)
        return activations

    def parameters(self):
        for layer in self.layers:
            yield from layer.parameters()
            

    def accumulate_gradients(self, gradients_forward):

        for layer in reversed(self.layers):
            gradients_forward = layer.accumulate_gradients(gradients_forward=gradients_forward)

    def __repr__(self):
            out = ""
            for k, layer in enumerate(self.layers):
                out += f"Layer {k}: {str(layer)}\n\n"
            return out
    
    def stats(self):
        header = f"{'Layer':<7}{'Inputs':>10}{'Neurons':>10}{'Params':>12}  {'Activation':<10}"
        line = "-" * len(header)

        print(line)
        print(header)
        print(line)

        total = 0
        for k, layer in enumerate(self.layers):
            params = layer.nout * (layer.nin + 1)   # weights + biases
            total += params
            act = "ReLU" if layer.nonlin else "Linear"
            print(f"{k:<7}{layer.nin:>10,}{layer.nout:>10,}{params:>12,}  {act:<10}")

        print(line)
        print(f"Layers: {len(self.layers)}")
        print(f"Total parameters: {total:,}")
        print(f"Memory (weights, float32): {total * 4 / 1e6:.2f} MB")
        print(line)


class SGD(Optimizer): # Standard Gradient Decent, dumb decent
    def __init__(self, model: Model, learning_rate):
        self.model = model
        self.learning_rate = learning_rate

    def step_accumulate_gradients(self, x, y): # accumulates allowing for chunking

        z = self.model(x)
        self.model.accumulate_gradients(z_loss_gradient(z=z, y=y))
        return z_loss(z=z, y=y)
    
    def step_learn(self):
        for p, g in self.model.parameters():
            g *= self.learning_rate              # in place, no temp
            p -= g   
        self.model.zero_grad()





def z_loss_gradient(z, y): # IMPORTANT : must be at least 2 classes or else gradient is always 0
    if len(z) == 1: raise ArithmeticError("Must have at least 2 outputs or else gradients are always 0")

    exp_z = np.exp(z - np.max(z))  # Subtract max to prevent overflow
    probs = exp_z / np.sum(exp_z)  # Softmax activation
    return probs - y

def z_loss(z, y):
    z_shifted = z - np.max(z)                      # for numerical stability
    log_probs = z_shifted - np.log(np.sum(np.exp(z_shifted)))   # log-softmax
    return -np.sum(y * log_probs)
    

In [26]:
correct = 0
test_loss = 0.0
for x, y in zip(X_test, Y_test):
    z = model(x)
    correct += z.argmax() == y.argmax()
    test_loss += z_loss(z, y)

print(f"test accuracy: {correct / len(X_test):.4f}  test loss: {test_loss / len(X_test):.4f}")

test accuracy: 0.9780  test loss: 0.0697


In [16]:
model = MLP(nin=14*14, nouts=[11*11, 8*8, 5*5, 10], )

optimizer = SGD(model=model, learning_rate=0.005)

trainer = Trainer(optimizer=optimizer, x_train_set=X_train, y_train_set=Y_train, chunk_size=12)

trainer.train(n_steps=5_000, report_freq=200)

step: 0 loss = 2.3601157665252686
step: 200 loss = 0.5085881948471069
step: 400 loss = 0.5769467353820801
step: 600 loss = 0.09457644075155258
step: 800 loss = 0.4687154293060303
step: 1000 loss = 0.5934872031211853
step: 1200 loss = 0.3699820935726166
step: 1400 loss = 0.3340126574039459
step: 1600 loss = 0.03976549208164215
step: 1800 loss = 0.018540197983384132
step: 2000 loss = 0.5079475045204163
step: 2200 loss = 0.27719905972480774
step: 2400 loss = 0.09293077141046524
step: 2600 loss = 0.38836023211479187
step: 2800 loss = 0.2521050274372101
step: 3000 loss = 0.2525746524333954
step: 3200 loss = 0.29081860184669495
step: 3400 loss = 0.04871481657028198
step: 3600 loss = 0.17092759907245636
step: 3800 loss = 0.07436969876289368
step: 4000 loss = 0.44895699620246887
step: 4200 loss = 0.018604114651679993
step: 4400 loss = 0.07615358382463455
step: 4600 loss = 0.009254113771021366
step: 4800 loss = 0.13843968510627747


In [25]:
optimizer.learning_rate= 0.0005
trainer.train(n_steps=5_000, report_freq=200)

step: 0 loss = 0.024304991587996483
step: 200 loss = 0.0005755097372457385
step: 400 loss = 0.009880377911031246
step: 600 loss = 0.0007763262838125229
step: 800 loss = 0.027072647586464882
step: 1000 loss = 0.20772522687911987
step: 1200 loss = 0.006133596878498793
step: 1400 loss = 0.07662755995988846
step: 1600 loss = 0.0026383260264992714
step: 1800 loss = 0.0013465496012941003
step: 2000 loss = 0.019014785066246986
step: 2200 loss = 0.021629078313708305
step: 2400 loss = 0.0053467098623514175
step: 2600 loss = 0.012547977268695831
step: 2800 loss = 0.012820354662835598
step: 3000 loss = 0.01866593398153782
step: 3200 loss = 0.002299841959029436
step: 3400 loss = 0.005103840958327055
step: 3600 loss = 0.005910927429795265
step: 3800 loss = 0.0014200069708749652
step: 4000 loss = 0.04226672276854515
step: 4200 loss = 0.0005018062074668705
step: 4400 loss = 0.04278618097305298
step: 4600 loss = 0.010909643955528736
step: 4800 loss = 0.04121642932295799
